In [4]:
import pandas as pd
import requests
import io

CARTO_SQL_URL = "https://phl.carto.com/api/v2/sql"

def carto_query(sql: str) -> pd.DataFrame:
    """Run a SQL query against Philadelphia's public CARTO SQL API and return a DataFrame."""
    resp = requests.get(
        CARTO_SQL_URL,
        params={"q": sql, "format": "csv", "api_key": "default_public"},
        headers={"User-Agent": "Mozilla/5.0 (compatible; research-notebook/1.0)"},
    )
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text))

In [9]:
CANDIDATE_TABLE_NAMES = ["public_cases_fc", "311_requests", "service_requests"]

found_table = None
for name in CANDIDATE_TABLE_NAMES:
    try:
        sample = carto_query(f"SELECT * FROM {name} LIMIT 5")
        found_table = name
        print(f"Found table: {name}\n")
        print(sample.columns.tolist())
        break
    except Exception as e:
        print(f"'{name}' failed: {e}")

if found_table is None:
    print(
        "None of the candidate table names worked.\n"
        "Go to https://opendataphilly.org and search '311' to find the current dataset "
        "page — it will list the CARTO table name (or a direct CSV export) under "
        "'API' / 'Data & Resources'. Paste the correct name into CANDIDATE_TABLE_NAMES above."
    )

Found table: public_cases_fc

['the_geom', 'cartodb_id', 'the_geom_webmercator', 'objectid', 'service_request_id', 'status', 'status_notes', 'service_name', 'service_code', 'agency_responsible', 'service_notice', 'requested_datetime', 'updated_datetime', 'expected_datetime', 'address', 'zipcode', 'media_url', 'lat', 'lon', 'subject', 'closed_datetime', 'gdb_geomattr_data', 'service_type']


In [10]:
sample = carto_query(f"SELECT * FROM {found_table} LIMIT 20")
sample.head(20)

,the_geom,cartodb_id,the_geom_webmercator,objectid,service_request_id,status,status_notes,service_name,service_code,agency_responsible,...,expected_datetime,address,zipcode,media_url,lat,lon,subject,closed_datetime,gdb_geomattr_data,service_type
0,0101000020E6100000022F4CE0EFCA52C08CB55F9016FF...,1,0101000020110F00000577E352DCEB5FC1FB80DFC6E28E...,1,9240889,Closed,Other,Alley Light Outage,SR-ST06,Streets Department,...,2015-08-10 00:00:00+00,2334 W CUMBERLAND ST,19132.0,NaN,39.992876,-75.170891,NaN,2026-08-26 15:52:39+00,NaN,Alley Light Outage
1,0101000020E6100000CB30EE06D1CA52C0268DD13AAAFE...,2,0101000020110F0000226C6AECA7EB5FC13FAE1BAF6A8E...,2,10007982,Closed,Question Answered,Information Request,SR-IR01,Philly311 Contact Center,...,NaN,2173–2215 W Dauphin St,19132.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.989570,-75.169008,NaN,2016-12-29 16:41:30+00,NaN,Information Request
2,0101000020E61000001C10217F32CE52C08332300EF103...,3,0101000020110F0000B461B50266F15FC1020C708B4494...,3,10014183,Closed,NaN,Maintenance Residential or Commercial,SR-LI21,License & Inspections,...,2016-02-10 00:00:00+00,303 GREEN LN,19128.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,40.030794,-75.221832,NaN,2016-01-23 02:00:48+00,NaN,Maintenance Residential
3,0101000020E6100000350D76DF05CB52C06B2B69B7E4F8...,4,0101000020110F00009F89FCAF01EC5FC158F075610588...,4,10030187,Closed,Repair Completed,Graffiti Removal,SR-CL01,Community Life Improvement Program,...,2016-01-20 00:00:00+00,1800 SOUTH ST,19146.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.944480,-75.172233,NaN,2016-01-13 20:24:16+00,NaN,Graffiti Removal
4,0101000020E61000000D4B1B64F7CA52C032BE3DDF51F9...,5,0101000020110F0000B2F7AB16E9EB5FC11155E74D7E88...,5,10034988,Closed,Repair Completed,Graffiti Removal,SR-CL01,Community Life Improvement Program,...,2016-01-20 00:00:00+00,1801 SPRUCE ST,19103.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.947811,-75.171350,NaN,2016-01-13 19:48:03+00,NaN,Graffiti Removal
5,NaN,6,NaN,6,10036132,Closed,NaN,Vacant House or Commercial,SR-LI27,License & Inspections,...,2014-02-03 00:00:00+00,02409 N CLEVELAND ST,19132.0,NaN,NaN,NaN,NaN,2014-01-02 09:17:38+00,NaN,Vacant House
6,NaN,7,NaN,7,10036135,Closed,Question Answered,Vacant House or Commercial,SR-LI27,License & Inspections,...,2015-11-09 00:00:00+00,1800 ringgold st,19146.0,NaN,NaN,NaN,NaN,2014-01-02 10:52:56+00,NaN,Vacant House
7,NaN,8,NaN,8,10036136,Closed,Question Answered,Building Dangerous,SR-LI03,License & Inspections,...,2014-01-24 00:00:00+00,1801 ringold st,19146.0,NaN,NaN,NaN,NaN,2014-01-02 10:53:51+00,NaN,Building Dangerous Vacant
8,0101000020E61000001E3BE26F3FC652C0D64B7C83E905...,9,0101000020110F0000B443D441E5E35FC156D5F6247496...,9,10036242,Closed,NaN,Infestation Residential,SR-LI17,License & Inspections,...,2014-02-06 00:00:00+00,06130 PALMETTO ST,19111.0,NaN,40.046189,-75.097622,NaN,2014-01-06 14:54:14+00,NaN,Infestation Residential
9,0101000020E610000040A75AEAA4C352C08AE895BAFD03...,10,0101000020110F00005526671A79DF5FC1FC30149A5294...,10,10036248,Closed,Question Answered,Zoning Residential,SR-LI30,License & Inspections,...,2015-11-09 00:00:00+00,03147 GILHAM ST,19149.0,NaN,40.031181,-75.056941,NaN,2014-01-06 17:23:27+00,NaN,Zoning Residential


In [14]:
type_counts[type_counts["service_name"].str.contains(
    "pothole|street light|traffic signal|street defect|abandoned vehicle",
    case=False, na=False
)]

,service_name,n
2,Abandoned Vehicle,275351
5,Street Defect,170339
7,Street Light Outage,141168
11,Traffic Signal Emergency,46799


In [18]:
CHOSEN_TYPES = ["Abandoned Vehicle"]

assert CHOSEN_TYPES

In [19]:

def fetch_all(table: str, type_col: str, types: list[str], page_size: int = 20000) -> pd.DataFrame:
    types_sql = ", ".join(f"'{t}'" for t in types)
    chunks = []
    offset = 0
    while True:
        sql = (
            f"SELECT * FROM {table} "
            f"WHERE {type_col} IN ({types_sql}) "
            f"ORDER BY objectid "
            f"LIMIT {page_size} OFFSET {offset}"
        )
        chunk = carto_query(sql)
        if len(chunk) == 0:
            break
        chunks.append(chunk)
        print(f"fetched rows {offset} - {offset + len(chunk)}")
        offset += page_size
        if len(chunk) < page_size:
            break
    return pd.concat(chunks, ignore_index=True)

df = fetch_all(found_table, TYPE_COLUMN, CHOSEN_TYPES)
df.shape


fetched rows 0 - 20000
fetched rows 20000 - 40000
fetched rows 40000 - 60000
fetched rows 60000 - 80000
fetched rows 80000 - 100000
fetched rows 100000 - 120000
fetched rows 120000 - 140000
fetched rows 140000 - 160000
fetched rows 160000 - 180000
fetched rows 180000 - 200000
fetched rows 200000 - 220000
fetched rows 220000 - 240000
fetched rows 240000 - 260000
fetched rows 260000 - 275351


(275351, 23)

In [20]:
print(df.shape)
print(df.dtypes)
print(df.isna().mean().sort_values(ascending=False))

(275351, 23)
the_geom                 object
cartodb_id                int64
the_geom_webmercator     object
objectid                  int64
service_request_id        int64
status                   object
status_notes             object
service_name             object
service_code             object
agency_responsible       object
service_notice           object
requested_datetime       object
updated_datetime         object
expected_datetime        object
address                  object
zipcode                  object
media_url                object
lat                     float64
lon                     float64
subject                 float64
closed_datetime          object
gdb_geomattr_data       float64
service_type             object
dtype: object
subject                 1.000000
gdb_geomattr_data       1.000000
media_url               0.739645
status_notes            0.349314
closed_datetime         0.083283
zipcode                 0.005680
service_notice          0.002887
the_ge

In [21]:
df.to_csv("philly311_raw.csv", index=False)
print("Saved philly311_raw.csv")

Saved philly311_raw.csv
